In [1]:
import json, re, subprocess, sys, zipfile
from collections import defaultdict
from pathlib import Path
import pandas as pd

def clone_repo(github_url: str, dest: str = "cloned_repo") -> Path:
    dest_path = Path(dest)
    if dest_path.exists():
        print(f"{dest_path} already exists, skipping clone.")
        return dest_path
    result = subprocess.run(["git", "clone", "--depth", "1", github_url, str(dest_path)],
                             capture_output=True, text=True)
    if result.returncode != 0:
        print("CLONE FAILED:", result.stderr); sys.exit(1)
    print(f"Cloned {github_url} -> {dest_path}")
    return dest_path

In [2]:
def find_gradle_modules(repo_dir: Path):
    files = list(repo_dir.rglob("build.gradle")) + list(repo_dir.rglob("build.gradle.kts"))
    return [f for f in files if "/.gradle/" not in str(f) and "/build/" not in str(f)]

DEP_LINE = re.compile(
    r"""(implementation|api|testImplementation|compileOnly|runtimeOnly|annotationProcessor)
        \s*\(?\s*['"](?P<coord>[^'"]+)['"]""", re.VERBOSE)

def parse_gradle_file(path: Path):
    text = path.read_text()
    deps = []
    for m in DEP_LINE.finditer(text):
        parts = m.group("coord").split(":")
        if len(parts) == 3:
            group, artifact, version = parts
        elif len(parts) == 2:
            group, artifact = parts; version = "MANAGED"
        else:
            continue
        deps.append({"scope": m.group(1), "group": group, "artifact": artifact,
                     "version": version, "coordinate": f"{group}:{artifact}"})
    return deps

def parse_all_modules(repo_dir: Path):
    modules = find_gradle_modules(repo_dir)
    print(f"Found {len(modules)} Gradle build file(s):")
    all_deps = []
    for mod in modules:
        print(f"  - {mod.relative_to(repo_dir)}")
        all_deps.extend(parse_gradle_file(mod))
    return all_deps

In [7]:
import zipfile, io, re
from pathlib import Path

def find_boot_jar(repo_dir):
    """Spring Boot's bootJar task outputs to build/libs/. Newer Spring Boot
    versions sometimes also produce a '-plain.jar' (classes only, no bundled
    deps) -- skip those, we want the executable fat jar."""
    libs_dir = Path(repo_dir) / "build" / "libs"
    if not libs_dir.exists():
        return None
    candidates = [j for j in libs_dir.glob("*.jar") if "-plain" not in j.name and "sources" not in j.name]
    return candidates[0] if candidates else None


def build_class_index_from_boot_jar(boot_jar_path):
    """Spring Boot already bundled every RESOLVED runtime dependency jar
    inside BOOT-INF/lib/. No global Gradle cache hunting needed -- open the
    outer jar, read each inner jar's bytes from memory, open THAT as a
    nested zip, and index its .class files."""
    class_index = {}
    artifact_to_version = {}

    with zipfile.ZipFile(boot_jar_path) as outer:
        lib_entries = [n for n in outer.namelist() if n.startswith("BOOT-INF/lib/") and n.endswith(".jar")]
        print(f"Found {len(lib_entries)} bundled dependency jars in {boot_jar_path.name}")

        for entry_name in lib_entries:
            jar_filename = entry_name.split("/")[-1]
            inner_bytes = outer.read(entry_name)
            m = re.match(r"^(.+?)-(\d[\w.\-]*)\.jar$", jar_filename)
            artifact = m.group(1) if m else jar_filename.replace(".jar", "")
            version = m.group(2) if m else "UNKNOWN"
            artifact_to_version[artifact] = version

            with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
                for name in inner.namelist():
                    if name.endswith(".class") and "/" in name and "module-info" not in name:
                        class_name = name[:-6].replace("/", ".")
                        class_index[class_name] = artifact

    return class_index, artifact_to_version


def resolve_groups_from_declared(class_index, declared_coords):
    """The jar filename only gives us 'artifact', not 'group:artifact'.
    Cross-reference against stage 1's declared dependency list to recover
    the full coordinate. Anything not found stays artifact-only -- it's a
    transitive dependency we never directly declared ourselves."""
    artifact_to_full = {coord.split(":")[1]: coord for coord in declared_coords}
    return {cls: artifact_to_full.get(artifact, f"UNKNOWN_GROUP:{artifact}")
            for cls, artifact in class_index.items()}

In [8]:
IMPORT_RE = re.compile(r"^\s*import\s+(?:static\s+)?([\w.]+)\s*;", re.MULTILINE)
CALL_RE = re.compile(r"\b([A-Za-z_][A-Za-z0-9_]*)\.([a-zA-Z0-9_]+)\s*\(")
VAR_DECL_RE = re.compile(r"\b([A-Z][A-Za-z0-9_]*)\s+([a-zA-Z_][A-Za-z0-9_]*)\s*[=;,)]")

def scan_source_with_jar_index(repo_dir: Path, class_index: dict):
    files = [f for f in list(repo_dir.rglob("*.java")) + list(repo_dir.rglob("*.kt"))
             if "/build/" not in str(f)]
    print(f"Scanning {len(files)} source file(s)")
    usage_count, usage_calls = defaultdict(int), defaultdict(set)
    java_time_used = False
    for f in files:
        text = f.read_text()
        imports, calls, var_decls = IMPORT_RE.findall(text), CALL_RE.findall(text), VAR_DECL_RE.findall(text)
        class_to_coord = {}
        for imp in imports:
            if imp.startswith("java.time"):
                java_time_used = True
            coord = class_index.get(imp)  # DIRECT exact lookup -- the whole point
            if coord:
                class_name = imp.split(".")[-1]
                class_to_coord[class_name] = coord
                usage_count[coord] += 1
        var_to_class = {var: cls for cls, var in var_decls if cls in class_to_coord}
        for identifier, method in calls:
            if identifier in class_to_coord:
                usage_calls[class_to_coord[identifier]].add(f"{identifier}.{method}")
            elif identifier in var_to_class:
                cls = var_to_class[identifier]
                usage_calls[class_to_coord[cls]].add(f"{cls}.{method}")
    return usage_count, usage_calls, java_time_used

In [9]:
def load_excel_playbook(path: str):
    """Returns (CATEGORY, playbook). Both are human-curated knowledge,
    maintained as 3 sheets in one workbook instead of editing Python/YAML:
      - library_categories: coordinate | category
      - categories: category | libraries | recommended_library | recommendation_reason | remove_libraries
      - method_mappings: category | from | from_library | to | behavioral_risk | risk_note | confidence | no_equivalent
    """
    lib_cat_df = pd.read_excel(path, sheet_name="library_categories")
    CATEGORY = dict(zip(lib_cat_df["coordinate"], lib_cat_df["category"]))

    categories_df = pd.read_excel(path, sheet_name="categories")
    mappings_df = pd.read_excel(path, sheet_name="method_mappings")

    playbook = {}
    for _, row in categories_df.iterrows():
        cat = row["category"]
        playbook[cat] = {
            "libraries": [s.strip() for s in str(row["libraries"]).split(",")],
            "recommended_library": row["recommended_library"],
            "recommendation_reason": row["recommendation_reason"],
            "remove_libraries": [s.strip() for s in str(row["remove_libraries"]).split(",")],
            "method_mappings": [],
        }
    mappings_by_cat = defaultdict(list)
    for _, row in mappings_df.iterrows():
        mappings_by_cat[row["category"]].append({
            "from": row["from"], "from_library": row["from_library"], "to": row["to"],
            "behavioral_risk": row["behavioral_risk"], "risk_note": row["risk_note"],
            "confidence": float(row["confidence"]), "no_equivalent": bool(row["no_equivalent"]),
        })
    for cat, mappings in mappings_by_cat.items():
        if cat in playbook:
            playbook[cat]["method_mappings"] = mappings
    return CATEGORY, playbook

# CATEGORY, playbook = load_excel_playbook("playbook.xlsx")

In [10]:
def run_pipeline(github_url: str, gradle_cache_dir: str, playbook_xlsx: str, dest: str = "cloned_repo"):
    repo_dir = clone_repo(github_url, dest)
    declared = parse_all_modules(repo_dir)
    declared_coords = {d["coordinate"] for d in declared}
    print(f"\nDeclared dependencies: {len(declared_coords)}")
    for c in sorted(declared_coords): print(f"  - {c}")

    jar_map = find_gradle_cache_jars(gradle_cache_dir)
    class_index = build_class_index_from_jars(jar_map)
    print(f"\nClass index built from {len(jar_map)} jars: {len(class_index)} classes indexed")

    usage_count, usage_calls, java_time_used = scan_source_with_jar_index(repo_dir, class_index)
    used = sorted(set(usage_count.keys()) & declared_coords)

    CATEGORY, playbook = load_excel_playbook(playbook_xlsx)

    clusters = defaultdict(list)
    for coord in used: clusters[CATEGORY.get(coord, "uncategorized")].append(coord)
    if java_time_used: clusters["datetime"].append("jdk:java.time")
    duplicate_clusters = {c: v for c, v in clusters.items() if len(v) > 1}

    print(f"\nUSED: {used}")
    print(f"DUPLICATE CLUSTERS: {duplicate_clusters or 'none found'}")

    all_observed = {c for calls in usage_calls.values() for c in calls}
    equivalence_report = {}
    for category, libs in duplicate_clusters.items():
        entry = playbook.get(category)
        if entry is None:
            print(f"GAP: '{category}' not in playbook -- {libs}")
            continue
        relevant = [m for m in entry["method_mappings"] if m["from"] in all_observed]
        equivalence_report[category] = {"recommended_library": entry["recommended_library"],
                                         "remove_libraries": [l for l in entry["remove_libraries"] if l in libs],
                                         "method_mappings": relevant}
    print(f"\nEQUIVALENCE REPORT:\n{json.dumps(equivalence_report, indent=2)}")
    return equivalence_report

# run_pipeline(
#     github_url="https://github.com/your-org/your-service.git",
#     gradle_cache_dir=str(Path.home() / ".gradle/caches/modules-2/files-2.1"),
#     playbook_xlsx="playbook.xlsx",
# )